<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/4_clase_acceso_a_bases_de_datos_NCBI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Acceso a las bases de datos del NCBI con Biopython

**Nota para Google Colab:**  
- Ejecuta la celda de instalación si es necesario (`!pip install biopython`).  
- Reemplaza el correo electrónico de ejemplo por uno real (NCBI lo recomienda).  
- Las consultas a Entrez requieren conexión a internet.

En este notebook veremos cómo acceder a las bases de datos del **National Center for Biotechnology Information (NCBI)**. No solo trataremos GenBank, sino también otras bases de datos del NCBI. Muchas personas se refieren (erróneamente) a todo el conjunto de bases de datos del NCBI como GenBank, pero el NCBI incluye la base de datos de nucleótidos y muchas otras, por ejemplo, PubMed.

## Verificar las bases de datos disponibles en el NCBI

Biopython proporciona una interfaz a **Entrez**, el sistema de recuperación de datos disponible en el NCBI. Entrez también se puede usar a través del navegador web: https://www.ncbi.nlm.nih.gov/search/

**CONSEJOS:**
- Especifica una dirección de correo electrónico con tu consulta
- Evita un gran número de solicitudes (100 o más) durante las horas pico (entre las 9:00 a.m. y las 5:00 p.m. hora del Este de EE. UU. en días laborables)
- No realices más de tres consultas por segundo (Biopython se encargará de esto por ti)

No solo es una buena práctica de ciudadanía digital, sino que corres el riesgo de ser bloqueado si abusas de los servidores del NCBI (una buena razón para proporcionar una dirección de correo real, porque el NCBI puede intentar contactarte).

In [ ]:
from Bio import Entrez, SeqIO

In [ ]:
# ¡Reemplaza esto por tu correo electrónico real!
Entrez.email = "dumy@example.com"

**EInfo:** obtiene una lista de todos los nombres de bases de datos accesibles a través de Entrez

In [ ]:
# Esto te da la lista de bases de datos disponibles
handle = Entrez.einfo()
rec = Entrez.read(handle)
handle.close()
print(rec.keys())

In [ ]:
rec['DbList']

Ahora intentaremos encontrar el gen del **transportador de resistencia a la cloroquina (CRT)** (KM288867) en **Plasmodium falciparum** (el parásito que causa la forma más mortal de malaria) en la base de datos de nucleótidos:

**ESearch:** Búsqueda en las bases de datos de Entrez

Ten en cuenta que la búsqueda estándar limitará el número de referencias de registros a **20**, por lo que si tenemos más, podemos anular **retmax** al número deseado de registros.

In [ ]:
handle = Entrez.esearch(db="nucleotide", term='CRT[Gene Name] AND "Plasmodium falciparum"[Organism]', retmax="40")
rec_list = Entrez.read(handle)
handle.close()
rec_list['Count']

In [ ]:
len(rec_list['IdList'])

In [ ]:
rec_list['IdList']

Ahora tenemos los IDs de todos los registros, pero aún necesitamos recuperar los registros correctamente.

**EFetch:** Descarga de registros completos desde Entrez

Solicitar un formato de archivo específico desde Entrez usando `Bio.Entrez.efetch()` requiere especificar los argumentos opcionales **rettype** y/o **retmode**. Las diferentes combinaciones se describen para cada tipo de base de datos en las páginas enlazadas en la página web de NCBI efetch: https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.EFetch

- `rettype` - tipo de retorno, `gb` == GenBank  
- `retmax` - Número total de registros del conjunto de entrada que se recuperarán, hasta un máximo de 10,000

In [ ]:
id_list = rec_list['IdList']
handle = Entrez.efetch(db='nucleotide', id=id_list, rettype='gb')  # formato GenBank, necesitamos parsearlo con el módulo SeqIO

In [ ]:
recs = list(SeqIO.parse(handle, 'gb'))
handle.close()

Observa que hemos convertido un iterador (el resultado de `SeqIO.parse`) en una lista. La ventaja de hacer esto es que podemos usar el resultado tantas veces como queramos (por ejemplo, iterar muchas veces), sin repetir la consulta en el servidor.

In [ ]:
len(recs)

Sin embargo, ten cuidado con esta técnica, porque recuperarás una gran cantidad de registros completos, y algunos de ellos tendrán secuencias bastante grandes en su interior. Corres el riesgo de descargar muchos datos (lo cual sería una carga tanto para tu lado como para los servidores del NCBI).

In [ ]:
for rec in recs:
    if rec.name == 'KM288867':  # intentar encontrar el gen CRT en los 40 registros que descargamos
        break
print(rec.name)
print(rec.description)

In [ ]:
str(rec.seq)

## Descarga de archivos PDB (estructura de proteínas)

Además de las bases de datos del NCBI, también podemos descargar estructuras de proteínas desde el [RCSB PDB](https://www.rcsb.org/).

In [ ]:
import requests

# Define el ID de PDB
pdb_id = "8S81"  # Reemplaza con el ID de PDB que desees

# Define la URL
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

# Envía la solicitud GET
response = requests.get(url)

# Verifica si la solicitud fue exitosa
if response.status_code == 200:
    # Guarda el archivo
    with open(f"{pdb_id}.pdb", "w") as file:
        file.write(response.text)
    print(f"¡Archivo {pdb_id}.pdb descargado exitosamente!")
else:
    print(f"Error al descargar el archivo PDB. Código de estado: {response.status_code}")


### Lectura de un archivo PDB con Biopython

In [ ]:
import requests
from Bio.PDB import PDBParser

# 1. Descargar el archivo PDB
pdb_id = "1CRN"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

response = requests.get(url)

if response.status_code == 200:
    with open(f"{pdb_id}.pdb", "w") as file:
        file.write(response.text)
    print(f"¡Archivo {pdb_id}.pdb descargado exitosamente!")
else:
    print(f"Error al descargar. Código: {response.status_code}")

# 2. Leer la estructura
parser = PDBParser()
structure = parser.get_structure(pdb_id, f"{pdb_id}.pdb")
print(structure)

In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser()
structure = parser.get_structure("1CRN", "1CRN.pdb")  # Asegúrate de tener el archivo 1CRN.pdb
print(structure)